# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.015 · Ampliación dirigida de categorías minoritarias

Descubre y adquiere videos peruanos nuevos para llevar cada categoría de daño a por lo menos 2.000 chunks en train, sin modificar el cuaderno 01_01 de scraping inicial.

La selección usa el snapshot efectivo después de las decisiones CODEX–Sol-EH y mide el déficit exclusivamente en `train`. El muestreo dirigido adapta principios de aprendizaje activo ante desbalance y clasificación multietiqueta de cola larga [1] [2]. Los candidatos reciben un split estable por `video_id` antes de descargar y etiquetar; la búsqueda no consulta etiquetas de validation o test para decidir la meta. Se reserva una fracción de adquisición para ambos holdouts, pero solo los chunks nuevos cuyo hash corresponde a train cuentan hacia el objetivo de 2.000. La adquisición escribe VTT mediante `yt-dlp` sin descargar audio o video [3] y mantiene `youtube-transcript-api` como respaldo [4]. Las consultas y `target_category` son mecanismos de recuperación, nunca etiquetas: todo chunk nuevo debe pasar por el prompt operativo y la jerarquía de revisión. Las transcripciones automáticas pueden contener sesgos dialectales [5], y el uso de la plataforma debe respetar sus términos y el análisis ético contextual [6] [7].

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Preflight

In [ ]:
from moderacion_peru.artifacts import artifact_status
show_result('Disponibilidad de artefactos', artifact_status(ROOT), tone='neutral')

## Parámetros de la meta train ≥ 2.000

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTROLES DEL SCRAPING: edite únicamente este bloque
# ══════════════════════════════════════════════════════════════════════════════
DISCOVER_NEW = True           # Esta campaña descubre fuentes nuevas dirigidas
FETCH_NEW = True              # True: obtiene subtítulos solo de videos aún no procesados
BACKFILL_MISSING_VTT = False  # El backfill general permanece en 01_01
DISCOVERY_MODE = "directed"   # Fijo: ampliación de daños minoritarios
TARGET_TRAIN_CHUNKS_PER_DAMAGE = 2_000
SPLIT_SEED = 20260805
DIRECTED_SPLIT_BUDGET = {"train": 450, "validation": 80, "test": 80}

MAX_NEW_VIDEOS = None         # None: incluye todos los pendientes; use un entero para un piloto
MAX_VTT_BACKFILL = None       # None: intenta todos los VTT faltantes; reanudable por archivo
NETWORK_BATCH_SIZE = 10       # llamadas nuevas por lote antes de una pausa larga
NETWORK_BATCH_PAUSE_SECONDS = 15.0
EXCLUDE_CHANNEL_ON_429 = True # difiere solo el canal afectado; continúa con los demás
RANDOMIZE_DOWNLOAD_QUEUE = True
DOWNLOAD_RANDOM_SEED = 20260806 # orden reproducible e intercalado por canal
MAX_VIDEOS_PER_CHANNEL = 400  # inspección amplia; no implica descargar todos
MAX_RESULTS_PER_QUERY = 80    # candidatos inspeccionados por consulta dirigida
MAX_DIRECTED_CANDIDATES = 610 # 450 train + 80 validation + 80 test
MAX_DIRECTED_SEED_CHANNELS = 16
MAX_EXPANDED_CHANNELS = 16    # canales nuevos inferidos desde búsquedas temáticas
MAX_VIDEOS_PER_EXPANDED_CHANNEL = 150

SUBTITLE_LANGUAGES = ("es-PE", "es-419", "es")
MIN_TRANSCRIPT_CHARACTERS = 200
USE_TRANSCRIPT_API_FALLBACK = True
YT_RETRIES = 3
YT_SLEEP_MIN_SECONDS = 2.5
YT_SLEEP_MAX_SECONDS = 10.0
YT_SOCKET_TIMEOUT_SECONDS = 30.0 # máximo por operación HTTP antes de reintentar/omitir
RESUME_DISCOVERY = True       # checkpoint atómico después de cada canal o consulta
STOP_ON_VIDEO_ERROR = False   # False: registra el fallo y continúa con el siguiente
SYNC_TRANSCRIPTS_BY_CHANNEL = True # checkpoint pequeño y sincronizable por canal
SYNC_VTT_BY_VIDEO = True      # checkpoint VTT crudo, deduplicado y sincronizable

# Reinicio destructivo recuperable: deje vacío normalmente. Para archivar los
# artefactos activos y reconstruir el dataset de videos desde cero, descomente:
RESET_VIDEO_DATASET = ""
# RESET_VIDEO_DATASET = "ARCHIVAR_Y_REINICIAR_DATASET_VIDEOS"

if TARGET_TRAIN_CHUNKS_PER_DAMAGE < 1:
    raise ValueError("TARGET_TRAIN_CHUNKS_PER_DAMAGE debe ser positivo")
if set(DIRECTED_SPLIT_BUDGET) != {"train", "validation", "test"}:
    raise ValueError("DIRECTED_SPLIT_BUDGET debe declarar train, validation y test")
if any(value < 0 for value in DIRECTED_SPLIT_BUDGET.values()):
    raise ValueError("Los presupuestos por split no pueden ser negativos")
if (MAX_DIRECTED_CANDIDATES is not None
        and sum(DIRECTED_SPLIT_BUDGET.values()) > MAX_DIRECTED_CANDIDATES):
    raise ValueError("El presupuesto por split supera MAX_DIRECTED_CANDIDATES")
if DISCOVERY_MODE not in {"seed", "directed", "both"}:
    raise ValueError("DISCOVERY_MODE debe ser seed, directed o both")
if ((MAX_NEW_VIDEOS is not None and MAX_NEW_VIDEOS < 0)
        or (MAX_VTT_BACKFILL is not None and MAX_VTT_BACKFILL < 0)
        or NETWORK_BATCH_SIZE < 1 or NETWORK_BATCH_PAUSE_SECONDS < 0
        or MAX_VIDEOS_PER_CHANNEL < 1 or MAX_RESULTS_PER_QUERY < 1
        or (MAX_DIRECTED_CANDIDATES is not None and MAX_DIRECTED_CANDIDATES < 1)
        or MAX_DIRECTED_SEED_CHANNELS < 1
        or MAX_EXPANDED_CHANNELS < 0 or MAX_VIDEOS_PER_EXPANDED_CHANNEL < 1):
    raise ValueError("Los límites de videos deben ser válidos")
if MIN_TRANSCRIPT_CHARACTERS < 1:
    raise ValueError("MIN_TRANSCRIPT_CHARACTERS debe ser positivo")
if RANDOMIZE_DOWNLOAD_QUEUE and not str(DOWNLOAD_RANDOM_SEED).strip():
    raise ValueError("DOWNLOAD_RANDOM_SEED no puede estar vacío")
if YT_SLEEP_MIN_SECONDS < 0 or YT_SLEEP_MAX_SECONDS < YT_SLEEP_MIN_SECONDS:
    raise ValueError("El intervalo de espera de yt-dlp no es válido")
if YT_SOCKET_TIMEOUT_SECONDS <= 0:
    raise ValueError("YT_SOCKET_TIMEOUT_SECONDS debe ser positivo")

show_summary('Configuración del scraping', {
    "discover_new": DISCOVER_NEW,
    "fetch_new": FETCH_NEW,
    "backfill_missing_vtt": BACKFILL_MISSING_VTT,
    "discovery_mode": DISCOVERY_MODE,
    "target_train_chunks_per_damage": TARGET_TRAIN_CHUNKS_PER_DAMAGE,
    "split_seed": SPLIT_SEED,
    "directed_split_budget": DIRECTED_SPLIT_BUDGET,
    "max_new_videos": MAX_NEW_VIDEOS,
    "max_vtt_backfill": MAX_VTT_BACKFILL,
    "network_batch_size": NETWORK_BATCH_SIZE,
    "network_batch_pause_seconds": NETWORK_BATCH_PAUSE_SECONDS,
    "exclude_channel_on_429": EXCLUDE_CHANNEL_ON_429,
    "randomize_download_queue": RANDOMIZE_DOWNLOAD_QUEUE,
    "download_random_seed": DOWNLOAD_RANDOM_SEED,
    "max_videos_per_channel": MAX_VIDEOS_PER_CHANNEL,
    "max_results_per_query": MAX_RESULTS_PER_QUERY,
    "max_directed_candidates": MAX_DIRECTED_CANDIDATES,
    "max_directed_seed_channels": MAX_DIRECTED_SEED_CHANNELS,
    "max_expanded_channels": MAX_EXPANDED_CHANNELS,
    "subtitle_languages": SUBTITLE_LANGUAGES,
    "min_transcript_characters": MIN_TRANSCRIPT_CHARACTERS,
    "transcript_api_fallback": USE_TRANSCRIPT_API_FALLBACK,
    "yt_socket_timeout_seconds": YT_SOCKET_TIMEOUT_SECONDS,
    "resume_discovery": RESUME_DISCOVERY,
    "sync_transcripts_by_channel": SYNC_TRANSCRIPTS_BY_CHANNEL,
    "sync_vtt_by_video": SYNC_VTT_BY_VIDEO,
    "reset_video_dataset_armed": bool(RESET_VIDEO_DATASET),
}, tone='neutral')

if RESET_VIDEO_DATASET:
    from moderacion_peru.acquisition import reset_active_video_dataset
    reset_result = reset_active_video_dataset(ROOT, RESET_VIDEO_DATASET)
    show_result('Dataset activo archivado; reconstrucción desde cero habilitada', reset_result, tone='warning')


## Canales y consultas para daños minoritarios

In [ ]:
# Fuentes priorizadas por rendimiento histórico en train.
# Las cuotas son máximos de candidatos inspeccionados y nunca etiquetas automáticas.
SEED_CHANNELS = []
SEED_SEARCH_QUERIES = []
DIRECTED_CHANNEL_CATALOG = [
    {"name": "Hablando Huevadas", "url": "https://www.youtube.com/@HablandoHuevadasOficial", "quota": 350, "target_category": "RACISMO_DISCRIMINACION|ATAQUE_POR_GENERO_IDENTIDAD"},
    {"name": "Goblinciano", "url": "https://www.youtube.com/@Goblinciano", "quota": 350, "target_category": "RACISMO_DISCRIMINACION|ATAQUE_POR_GENERO_IDENTIDAD"},
    {"name": "Juanito y Richard", "url": "https://www.youtube.com/@JuanitoyRichard", "quota": 250, "target_category": "RACISMO_DISCRIMINACION|ATAQUE_POR_GENERO_IDENTIDAD"},
    {"name": "Nunca MAS", "url": "https://www.youtube.com/channel/UCFqwxsa2Wp6Y5FkUAcGShGA", "quota": 200, "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"name": "PBO", "url": "https://www.youtube.com/channel/UCgR0st4ZLABi-LQcWNu3wnQ", "quota": 150, "target_category": "RACISMO_DISCRIMINACION|ATAQUE_POR_GENERO_IDENTIDAD"},
    {"name": "Magaly TV La Firme", "url": "https://www.youtube.com/@MagalyTVLaFirmeATV", "quota": 150, "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"name": "Arde Troya con Juliana Oxenford", "url": "https://www.youtube.com/@ardetroyalr", "quota": 150, "target_category": "RACISMO_DISCRIMINACION|ATAQUE_POR_GENERO_IDENTIDAD"},
    {"name": "Todo Good", "url": "https://www.youtube.com/@todogoodpe", "quota": 150, "target_category": "RACISMO_DISCRIMINACION|ATAQUE_POR_GENERO_IDENTIDAD"},
]
DIRECTED_QUERY_CATALOG = [
    {"query": "podcast peruano insulto cholo serrano indio clasismo", "target_category": "RACISMO_DISCRIMINACION"},
    {"query": "comediante peruano racismo clasismo broma", "target_category": "RACISMO_DISCRIMINACION"},
    {"query": "streaming peruano burla regional conero provinciano", "target_category": "RACISMO_DISCRIMINACION"},
    {"query": "podcast Perú ataque por género identidad burla", "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"query": "streaming peruano burla machista misógina homofóbica", "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"query": "comediante peruano ataque machista homofóbico", "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
]


## Snapshot efectivo y déficit por chunks de train

`needs_review` de Pro es un estado intermedio. Una decisión posterior CODEX–Sol-EH o humana lo sustituye; si CODEX no cambió una propuesta Pro no vacía, prevalece Pro. El plan usa la última decisión efectiva, conserva el split histórico por video y calcula cuánto falta para 2.000 asignaciones en cada daño de train. Los canales se ordenan por rendimiento histórico, pero se combinan varias fuentes para evitar que una clase aprenda únicamente el estilo de un canal.

In [ ]:
import importlib
import moderacion_peru.acquisition as acquisition_module
importlib.reload(acquisition_module)

from moderacion_peru.acquisition import (
    VIDEO_DATASET_RESET_MARKER,
    build_directed_sampling_plan,
    collect_project_video_inventory,
    consolidate_available_transcripts,
    load_candidates,
    load_vtt_backfill_candidates,
    materialize_vtt_checkpoint,
    materialize_transcripts_by_channel,
    merge_candidates,
    select_directed_search_queries,
    select_directed_seed_channels,
)
from moderacion_peru.datasets import project_effective_training_rows
from moderacion_peru.io import read_jsonl
from moderacion_peru.taxonomy import load_taxonomy

CANONICAL = ROOT/'datos/raw/transcripts_raw.jsonl'
CACHE = ROOT/'datos/raw/transcripts_cache'
TRANSCRIPTS_BY_CHANNEL = ROOT/'datos/raw/transcripts_by_channel'
VTT_BY_VIDEO = ROOT/'datos/raw/vtt_by_video'
DIRECTED_DATASET = ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
DIRECTED_CAMPAIGN = ROOT/'datos/etiquetado/consolidado/anotaciones_v2.jsonl'
DIRECTED_REVIEWS = ROOT/'datos/etiquetado/humano/labeling_events_v2.jsonl'
previous_snapshot_rows = list(read_jsonl(DIRECTED_DATASET)) if DIRECTED_DATASET.exists() else []
if DIRECTED_CAMPAIGN.exists():
    DIRECTED_TRAINING_ROWS = project_effective_training_rows(
        read_jsonl(DIRECTED_CAMPAIGN),
        read_jsonl(DIRECTED_REVIEWS) if DIRECTED_REVIEWS.exists() else [],
        previous_snapshot_rows,
        seed=SPLIT_SEED,
    )
else:
    DIRECTED_TRAINING_ROWS = previous_snapshot_rows
rebuild_from_zero = (ROOT/VIDEO_DATASET_RESET_MARKER).exists()
consolidation_stats = consolidate_available_transcripts(
    ROOT,
    CANONICAL,
    cache_dir=CACHE,
    channel_dir=TRANSCRIPTS_BY_CHANNEL,
    vtt_dir=VTT_BY_VIDEO if VTT_BY_VIDEO.is_dir() else None,
    include_historical_snapshots=not rebuild_from_zero,
)
consolidation_stats['historical_bootstrap_disabled'] = rebuild_from_zero
show_result(
    'Consolidación de todas las transcripciones disponibles',
    consolidation_stats,
    tone='warning' if rebuild_from_zero else 'success',
)
if SYNC_TRANSCRIPTS_BY_CHANNEL:
    channel_partition_stats = materialize_transcripts_by_channel(CANONICAL, TRANSCRIPTS_BY_CHANNEL)
    show_summary('Checkpoint de transcripciones por canal', {
        'videos': channel_partition_stats['total_videos'],
        'canales': channel_partition_stats['total_channels'],
        'partes_jsonl': channel_partition_stats['total_channel_files'],
        'máximo_bytes_por_parte': channel_partition_stats['max_channel_file_bytes'],
        'carpeta': TRANSCRIPTS_BY_CHANNEL,
        'canonico_preservado': CANONICAL.exists(),
    }, tone='success')

VTT_BACKFILL_CANDIDATES = []
if SYNC_VTT_BY_VIDEO:
    vtt_checkpoint_stats = materialize_vtt_checkpoint(
        ROOT,
        VTT_BY_VIDEO,
        read_jsonl(CANONICAL) if CANONICAL.exists() else [],
    )
    VTT_BACKFILL_CANDIDATES = load_vtt_backfill_candidates(VTT_BY_VIDEO)
    show_summary('Checkpoint consolidado de VTT por video', {
        'archivos_vtt': vtt_checkpoint_stats['total_files'],
        'videos_con_vtt': vtt_checkpoint_stats['total_videos'],
        'videos_con_transcripcion': vtt_checkpoint_stats['transcript_videos'],
        'videos_sin_vtt': vtt_checkpoint_stats['missing_vtt_videos'],
        'vtt_invalidos': vtt_checkpoint_stats['invalid_vtt_files'],
        'cola_backfill': VTT_BY_VIDEO/'missing_vtt.jsonl',
        'carpeta': VTT_BY_VIDEO,
    }, tone='warning' if VTT_BACKFILL_CANDIDATES else 'success')

KNOWN_VIDEO_IDS, VIDEO_INVENTORY = collect_project_video_inventory(
    ROOT,
    canonical_path=CANONICAL,
    cache_dir=CACHE,
    include_historical_sources=not rebuild_from_zero,
    include_derived_sources=not rebuild_from_zero,
)
show_summary('Inventario global para evitar duplicaciones', {
    'transcripciones_canónicas_completas': VIDEO_INVENTORY['canonical_transcripts'],
    'transcripciones_completas_disponibles': VIDEO_INVENTORY['full_transcripts_union'],
    'videos_con_texto_derivado': VIDEO_INVENTORY['derived_text_videos'],
    'videos_solo_en_derivados': VIDEO_INVENTORY['derived_only_videos'],
    'videos_conocidos_globales': VIDEO_INVENTORY['known_videos_union'],
    'fuentes_raw_históricas': VIDEO_INVENTORY['historical_source_files'],
    'fuentes_derivadas': VIDEO_INVENTORY['derived_source_files'],
}, tone='warning' if VIDEO_INVENTORY['derived_only_videos'] else 'success')

taxonomy = load_taxonomy(ROOT/'config/taxonomia_v2.json')
directed_plan = None
DIRECTED_CHANNELS = []
DIRECTED_SEARCH_QUERIES = []
if DISCOVERY_MODE in {'directed', 'both'}:
    directed_plan = build_directed_sampling_plan(
        DIRECTED_TRAINING_ROWS,
        read_jsonl(CANONICAL) if CANONICAL.exists() else [],
        damage_labels=taxonomy.damage_labels,
        eligible_splits=('train',),
        target_chunks_per_label=TARGET_TRAIN_CHUNKS_PER_DAMAGE,
    )
    DIRECTED_CHANNELS = select_directed_seed_channels(
        directed_plan,
        DIRECTED_CHANNEL_CATALOG,
        max_channels=MAX_DIRECTED_SEED_CHANNELS,
    )
    DIRECTED_SEARCH_QUERIES = select_directed_search_queries(
        directed_plan,
        DIRECTED_QUERY_CATALOG,
        max_queries=len(DIRECTED_QUERY_CATALOG),
        max_results_per_query=MAX_RESULTS_PER_QUERY,
    )
    show_summary('Plan de ampliación dirigida', {
        'estrategia': directed_plan['strategy'],
        'objetivo_chunks_por_daño_train': directed_plan['target_chunks_per_label'],
        'soporte_chunks_train': directed_plan['support_chunks'],
        'déficit_chunks_train': directed_plan['deficit_chunks'],
        'adquisición_necesaria': directed_plan['acquisition_needed'],
        'pesos': {key: round(value, 4) for key, value in directed_plan['weights'].items()},
        'canales_semilla': len(DIRECTED_CHANNELS),
        'consultas_temáticas': len(DIRECTED_SEARCH_QUERIES),
    }, tone='warning' if directed_plan['strategy'] == 'fallback_equal' else 'success')

CHANNEL_SOURCES = (
    SEED_CHANNELS if DISCOVERY_MODE == 'seed'
    else DIRECTED_CHANNELS if DISCOVERY_MODE == 'directed'
    else SEED_CHANNELS + DIRECTED_CHANNELS
)
SEARCH_QUERIES = (
    SEED_SEARCH_QUERIES if DISCOVERY_MODE == 'seed'
    else DIRECTED_SEARCH_QUERIES if DISCOVERY_MODE == 'directed'
    else SEED_SEARCH_QUERIES + DIRECTED_SEARCH_QUERIES
)
show_summary('Fuentes seleccionadas', {
    'modo': DISCOVERY_MODE,
    'canales': len(CHANNEL_SOURCES),
    'consultas': len(SEARCH_QUERIES),
}, tone='neutral')


## Descubrimiento y cohorte separada por split

In [ ]:
from collections import Counter
import importlib
from tqdm.auto import tqdm
import moderacion_peru.acquisition as acquisition_module
importlib.reload(acquisition_module)

from moderacion_peru.acquisition import (
    discover_youtube_candidates,
    expand_directed_channel_sources,
    processed_video_ids,
    select_directed_candidates,
)
from moderacion_peru.io import append_jsonl_once, write_json_atomic, write_jsonl_atomic

DISCOVERED_PATH = ROOT/'datos/raw/video_candidates.jsonl'
DISCOVERY_FAILURES_PATH = ROOT/'datos/raw/fallos_descubrimiento_ultima_ejecucion.json'
DIRECTED_SELECTION_PATH = ROOT/'datos/raw/directed_candidates_latest.jsonl'
DIRECTED_PLAN_PATH = ROOT/'datos/raw/manifests/directed_plan_latest.json'
DISCOVERY_CHECKPOINT_PATH = ROOT/f'datos/raw/manifests/discovery_{DISCOVERY_MODE}_checkpoint.json'
discovered = []
directed_selection = []
expanded_channels = []
if DISCOVER_NEW:
    source_outcomes = Counter()
    source_total = len(CHANNEL_SOURCES) + len(SEARCH_QUERIES)
    source_progress = tqdm(total=source_total, desc='Descubriendo fuentes', unit='fuente')

    def report_discovery(event):
        source_name = str(event.get('source') or '(fuente sin nombre)')
        if event['status'] == 'started':
            source_progress.set_description(f'Descubriendo · {source_name[:42]}')
            source_progress.set_postfix(
                fuente=source_name[:42],
                correctas=source_outcomes['ok'],
                fallidas=source_outcomes['failed'],
                reanudadas=source_outcomes['resumed'],
                candidatos=event['candidates_unique'],
            )
            return
        source_outcomes[event['status']] += 1
        if event.get('resumed'):
            source_outcomes['resumed'] += 1
        source_progress.update(1)
        source_progress.set_postfix(
            fuente=source_name[:42],
            correctas=source_outcomes['ok'],
            fallidas=source_outcomes['failed'],
            reanudadas=source_outcomes['resumed'],
            candidatos=event['candidates_unique'],
        )

    try:
        discovered, discovery_failures = discover_youtube_candidates(
            CHANNEL_SOURCES,
            SEARCH_QUERIES,
            max_videos_per_channel=MAX_VIDEOS_PER_CHANNEL,
            max_results_per_query=MAX_RESULTS_PER_QUERY,
            retries=YT_RETRIES,
            sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
            sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
            socket_timeout_seconds=YT_SOCKET_TIMEOUT_SECONDS,
            checkpoint_path=DISCOVERY_CHECKPOINT_PATH if RESUME_DISCOVERY else None,
            progress_callback=report_discovery,
        )
        if directed_plan is not None:
            expanded_channels = expand_directed_channel_sources(
                discovered,
                directed_plan,
                known_channel_ids=[source.get('channel_id') for source in DIRECTED_CHANNELS],
                max_channels=MAX_EXPANDED_CHANNELS,
                videos_per_channel=MAX_VIDEOS_PER_EXPANDED_CHANNEL,
            )
            if expanded_channels:
                source_total += len(expanded_channels)
                source_progress.total = source_total
                source_progress.refresh()
                expanded_candidates, expanded_failures = discover_youtube_candidates(
                    expanded_channels,
                    (),
                    max_videos_per_channel=MAX_VIDEOS_PER_EXPANDED_CHANNEL,
                    max_results_per_query=MAX_RESULTS_PER_QUERY,
                    retries=YT_RETRIES,
                    sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
                    sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
                    socket_timeout_seconds=YT_SOCKET_TIMEOUT_SECONDS,
                    checkpoint_path=DISCOVERY_CHECKPOINT_PATH if RESUME_DISCOVERY else None,
                    progress_callback=report_discovery,
                )
                discovered = merge_candidates(discovered, expanded_candidates)
                discovery_failures.extend(expanded_failures)
            directed_pool = [
                candidate for candidate in discovered
                if candidate.get('sampling_mode') == 'directed'
            ]
            directed_known_excluded = len({
                str(candidate.get('video_id') or '').strip()
                for candidate in directed_pool
            } & KNOWN_VIDEO_IDS)
            selection_limit = (
                len(directed_pool)
                if MAX_DIRECTED_CANDIDATES is None
                else MAX_DIRECTED_CANDIDATES
            )
            directed_selection = []
            remaining_selection = selection_limit
            for planned_split, requested in DIRECTED_SPLIT_BUDGET.items():
                split_selection = select_directed_candidates(
                    directed_pool,
                    KNOWN_VIDEO_IDS,
                    directed_plan,
                    max_candidates=min(requested, remaining_selection),
                    required_split=planned_split,
                    split_seed=SPLIT_SEED,
                )
                directed_selection = merge_candidates(directed_selection, split_selection)
                remaining_selection -= len(split_selection)
            directed_selection = [
                {**candidate, 'directed_selection_rank': index}
                for index, candidate in enumerate(directed_selection, 1)
            ]
            directed_split_counts = dict(Counter(
                candidate.get('planned_split') for candidate in directed_selection
            ))
            write_jsonl_atomic(DIRECTED_SELECTION_PATH, directed_selection)
            write_json_atomic(DIRECTED_PLAN_PATH, {
                **directed_plan,
                'planned_channels': DIRECTED_CHANNELS,
                'planned_queries': DIRECTED_SEARCH_QUERIES,
                'expanded_channels': expanded_channels,
                'directed_candidates_discovered': len(directed_pool),
                'directed_known_videos_excluded': directed_known_excluded,
                'directed_candidates_selected': len(directed_selection),
                'directed_split_counts': directed_split_counts,
                'target_train_chunks_per_damage': TARGET_TRAIN_CHUNKS_PER_DAMAGE,
                'selection_path': DIRECTED_SELECTION_PATH,
            })
    finally:
        source_progress.close()
    added, existing = append_jsonl_once(DISCOVERED_PATH, discovered, id_field='video_id')
    write_json_atomic(DISCOVERY_FAILURES_PATH, discovery_failures)
    show_summary('Resumen del descubrimiento', {
        "sources_total": source_total,
        "sources_ok": source_outcomes['ok'],
        "sources_failed": source_outcomes['failed'],
        "sources_resumed": source_outcomes['resumed'],
        "candidates_unique": len(discovered),
        "candidates_added": added,
        "candidates_existing": existing,
        "expanded_channels": len(expanded_channels),
        "directed_cohort": len(directed_selection),
        "directed_split_counts": (directed_split_counts if directed_plan is not None else {}),
        "directed_known_videos_excluded": (
            directed_known_excluded if directed_plan is not None else 0
        ),
        "checkpoint": DISCOVERY_CHECKPOINT_PATH if RESUME_DISCOVERY else None,
    }, tone='success' if not discovery_failures else 'warning')
    if discovery_failures:
        failure_counts = Counter(row['failure_kind'] for row in discovery_failures)
        show_summary('Fallos de fuentes por motivo', dict(sorted(failure_counts.items())), tone='warning')
        show_summary('Artefacto de auditoría', {'ruta': DISCOVERY_FAILURES_PATH}, tone='neutral')
else:
    show_callout('Descubrimiento desactivado', 'Se reutilizan candidatos y transcripciones locales.', tone='neutral')


## Cohorte activa y caché

In [ ]:
import importlib
import moderacion_peru.acquisition as acquisition_module
importlib.reload(acquisition_module)

from moderacion_peru.acquisition import order_candidates_for_acquisition, processed_video_ids

if DISCOVERY_MODE == 'directed':
    candidates = load_candidates(DIRECTED_SELECTION_PATH)
    candidate_origin = 'cohorte_dirigida_vigente'
elif DISCOVERY_MODE == 'both' and DISCOVER_NEW:
    seed_candidates_current = [
        candidate for candidate in discovered
        if candidate.get('sampling_mode') == 'seed'
    ]
    candidates = merge_candidates(seed_candidates_current, directed_selection)
    candidate_origin = 'descubrimiento_actual_seed_más_cohorte_dirigida'
else:
    candidate_files = [
        ROOT/'datos/raw/video_candidates.jsonl',
        ROOT/'datos/raw/videos_candidatos.csv',
    ]
    candidates = merge_candidates(*(load_candidates(source) for source in candidate_files))
    candidate_origin = 'archivo_acumulado_general'

canonical_ids = processed_video_ids(CANONICAL)
candidate_ids = {str(candidate['video_id']).strip() for candidate in candidates}
existing_canonical_ids = candidate_ids & canonical_ids
existing_derived_ids = candidate_ids & (KNOWN_VIDEO_IDS - canonical_ids)
existing_known_ids = candidate_ids & KNOWN_VIDEO_IDS
pending_candidates = [
    candidate for candidate in candidates
    if str(candidate['video_id']).strip() not in KNOWN_VIDEO_IDS
]
if RANDOMIZE_DOWNLOAD_QUEUE:
    pending_candidates = order_candidates_for_acquisition(
        pending_candidates,
        random_seed=DOWNLOAD_RANDOM_SEED,
    )
cached_ids = {path.stem for path in CACHE.glob('*.json')}
pending_cached = sum(
    str(candidate['video_id']).strip() in cached_ids for candidate in pending_candidates
)
show_summary('Candidatos filtrados antes de la adquisición', {
    'origen': candidate_origin,
    'únicos': len(candidates),
    'transcripciones_canónicas_totales': len(canonical_ids),
    'videos_conocidos_globales': len(KNOWN_VIDEO_IDS),
    'ya_canónicos_omitidos': len(existing_canonical_ids),
    'ya_derivados_históricos_omitidos': len(existing_derived_ids),
    'ya_conocidos_omitidos_total': len(existing_known_ids),
    'pendientes_totales': len(pending_candidates),
    'pendientes_reutilizables_desde_caché': pending_cached,
    'pendientes_que_requieren_red': len(pending_candidates) - pending_cached,
    'cola_pseudoaleatoria': RANDOMIZE_DOWNLOAD_QUEUE,
    'semilla_cola': DOWNLOAD_RANDOM_SEED if RANDOMIZE_DOWNLOAD_QUEUE else None,
}, tone='neutral')


## Ejecución controlada y tolerante a fallos

In [ ]:
from functools import partial
import importlib
from tqdm.auto import tqdm
import moderacion_peru.acquisition as acquisition_module
importlib.reload(acquisition_module)

from moderacion_peru.acquisition import (
    backfill_missing_vtt,
    fetch_youtube_subtitles,
    ingest_incremental,
    materialize_transcripts_by_channel,
    materialize_vtt_checkpoint,
    order_candidates_for_acquisition,
)

FAILURES = ROOT/'datos/raw/fallos_adquisicion.jsonl'
VTT_FAILURES = ROOT/'datos/raw/fallos_vtt_backfill.jsonl'
fetcher = partial(
    fetch_youtube_subtitles,
    languages=SUBTITLE_LANGUAGES,
    retries=YT_RETRIES,
    sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
    sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
    socket_timeout_seconds=YT_SOCKET_TIMEOUT_SECONDS,
    minimum_transcript_characters=MIN_TRANSCRIPT_CHARACTERS,
    use_transcript_api_fallback=USE_TRANSCRIPT_API_FALLBACK,
    vtt_output_dir=VTT_BY_VIDEO if SYNC_VTT_BY_VIDEO else None,
)

vtt_backfill_queue = list(VTT_BACKFILL_CANDIDATES)
if RANDOMIZE_DOWNLOAD_QUEUE:
    vtt_backfill_queue = order_candidates_for_acquisition(
        vtt_backfill_queue,
        random_seed=DOWNLOAD_RANDOM_SEED,
    )
if BACKFILL_MISSING_VTT and vtt_backfill_queue:
    vtt_progress = tqdm(total=len(vtt_backfill_queue), desc='Recuperando VTT faltantes', unit='video')

    def report_vtt_backfill(event):
        counters = event['counters']
        vtt_progress.update(event.get('advance', 1))
        vtt_progress.set_description(
            'Pausa entre lotes VTT' if event['status'] == 'batch_pause' else 'Recuperando VTT faltantes'
        )
        vtt_progress.set_postfix(
            recuperados=counters['fetched'],
            fallidos=counters['failed'],
            diferidos=counters['deferred_by_limit'],
            pausa_429=counters['deferred_rate_limit'],
            canales_429=counters['rate_limited_channels'],
            lotes=counters['batch_pauses'],
        )

    try:
        vtt_backfill_stats = backfill_missing_vtt(
            vtt_backfill_queue,
            VTT_BY_VIDEO,
            fetcher=fetcher if FETCH_NEW else None,
            failure_path=VTT_FAILURES,
            max_new_videos=MAX_VTT_BACKFILL,
            network_batch_size=NETWORK_BATCH_SIZE,
            batch_pause_seconds=NETWORK_BATCH_PAUSE_SECONDS,
            exclude_rate_limited_channels=EXCLUDE_CHANNEL_ON_429,
            stop_on_error=STOP_ON_VIDEO_ERROR,
            progress_callback=report_vtt_backfill,
        )
    finally:
        vtt_progress.close()
    show_summary(
        'Resumen de recuperación VTT',
        {'pendientes_antes': len(vtt_backfill_queue), **vtt_backfill_stats},
        tone='success' if not vtt_backfill_stats['failed'] else 'warning',
    )
elif vtt_backfill_queue:
    show_callout(
        'Backfill VTT pendiente',
        f'Hay {len(vtt_backfill_queue)} videos sin VTT; active BACKFILL_MISSING_VTT y FETCH_NEW.',
        tone='warning',
    )
else:
    show_callout('VTT completos', 'No hay transcripciones canónicas sin VTT.', tone='success')

if pending_candidates:
    video_progress = tqdm(total=len(pending_candidates), desc='Procesando pendientes', unit='video')

    def report_acquisition(event):
        counters = event['counters']
        video_progress.update(event.get('advance', 1))
        if event['status'] == 'batch_pause':
            description = 'Pausa entre lotes'
        else:
            description = 'Procesando pendientes'
        video_progress.set_description(description)
        video_progress.set_postfix(
            existentes=counters['already_canonical'],
            cache=counters['reused_cache'],
            nuevos=counters['fetched'],
            fallidos=counters['failed'],
            diferidos=counters['deferred_by_limit'],
            pausa_429=counters['deferred_rate_limit'],
            intentos_429=counters.get('failure_rate_limited', 0),
            canales_429=counters['rate_limited_channels'],
            lotes=counters['batch_pauses'],
        )

    try:
        stats = ingest_incremental(
            pending_candidates,
            CANONICAL,
            CACHE,
            fetcher=fetcher if FETCH_NEW else None,
            failure_path=FAILURES,
            max_new_videos=MAX_NEW_VIDEOS,
            network_batch_size=NETWORK_BATCH_SIZE,
            batch_pause_seconds=NETWORK_BATCH_PAUSE_SECONDS,
            exclude_rate_limited_channels=EXCLUDE_CHANNEL_ON_429,
            stop_on_error=STOP_ON_VIDEO_ERROR,
            progress_callback=report_acquisition,
            channel_transcript_dir=TRANSCRIPTS_BY_CHANNEL if SYNC_TRANSCRIPTS_BY_CHANNEL else None,
        )
    finally:
        video_progress.close()
    stats = {
        'candidates_total': len(candidates),
        'filtered_existing_before_run': len(existing_known_ids),
        'pending_before_run': len(pending_candidates),
        **stats,
    }
    show_summary('Resumen de adquisición', stats, tone='success' if not stats['failed'] else 'warning')
    if stats['failed']:
        show_summary('Fallos de adquisición por motivo', {
            key.removeprefix('failure_'): value
            for key, value in stats.items()
            if key.startswith('failure_') and not key.startswith('failure_records_') and value
        }, tone='warning')
        show_summary('Videos omitidos sin detener el lote', {
            'cantidad': stats['failed'],
            'detalle': FAILURES,
        }, tone='warning')
elif candidates:
    show_callout(
        'Sin videos pendientes',
        'Todos los candidatos ya tienen una transcripción canónica; no se iniciaron descargas.',
        tone='success',
    )
else:
    show_callout(
        'No hay candidatos',
        'Active DISCOVER_NEW o añada un CSV/JSONL. El corpus existente no se vuelve a descargar.',
        tone='warning',
    )

if SYNC_TRANSCRIPTS_BY_CHANNEL:
    final_channel_stats = materialize_transcripts_by_channel(CANONICAL, TRANSCRIPTS_BY_CHANNEL)
    show_summary('JSONL por canal consolidados al cierre', {
        'videos': final_channel_stats['total_videos'],
        'canales': final_channel_stats['total_channels'],
        'partes_jsonl': final_channel_stats['total_channel_files'],
    }, tone='success')
if SYNC_VTT_BY_VIDEO:
    final_vtt_stats = materialize_vtt_checkpoint(
        ROOT,
        VTT_BY_VIDEO,
        read_jsonl(CANONICAL) if CANONICAL.exists() else [],
    )
    show_summary('VTT consolidados al cierre', {
        'archivos_vtt': final_vtt_stats['total_files'],
        'videos_con_vtt': final_vtt_stats['total_videos'],
        'videos_sin_vtt': final_vtt_stats['missing_vtt_videos'],
        'manifiesto_faltantes': VTT_BY_VIDEO/'missing_vtt.jsonl',
    }, tone='success' if not final_vtt_stats['missing_vtt_videos'] else 'warning')


## Referencias

[1] Y. Fairstein, O. Kalinsky, Z. Karnin, et al., "Class Balancing for Efficient Active Learning in Imbalanced Datasets," in Proc. 18th Linguistic Annotation Workshop, 2024, pp. 77–86, doi: 10.18653/v1/2024.law-1.8.

[2] Y. Huang, B. Giledereli, A. Köksal, et al., "Balancing Methods for Multi-label Text Classification with Long-Tailed Class Distribution," in Proc. EMNLP, 2021, pp. 8153–8161, doi: 10.18653/v1/2021.emnlp-main.643.

[3] yt-dlp contributors, "yt-dlp: A Feature-Rich Command-Line Audio/Video Downloader," GitHub repository, 2026. [Online]. Available: https://github.com/yt-dlp/yt-dlp. Accessed: Aug. 5, 2026.

[4] J. Depoix and contributors, "YouTube Transcript API: Python API for Retrieving YouTube Transcripts and Subtitles," GitHub repository, 2026. [Online]. Available: https://github.com/jdepoix/youtube-transcript-api. Accessed: Aug. 6, 2026.

[5] R. Tatman, "Gender and Dialect Bias in YouTube's Automatic Captions," in Proc. 1st ACL Workshop Ethics NLP, 2017, pp. 53–59, doi: 10.18653/v1/W17-1606.

[6] YouTube, "Terms of Service," Nov. 2023. [Online]. Available: https://www.youtube.com/t/terms. Accessed: Aug. 5, 2026.

[7] A. S. franzke, A. Bechmann, M. Zimmer, et al., "Internet Research: Ethical Guidelines 3.0," Association of Internet Researchers, 2020. [Online]. Available: https://aoir.org/reports/ethics3.pdf